In [ ]:
import asyncio

import numpy as np
from assistant.audio_handlers.openai_live import OpenAILive
from assistant.utils import AudioChunk, load_greeting
from dotenv import load_dotenv

load_dotenv()

OUT_DIR = "/Users/jckpn-work/dev/assistant-redux/src/assistant/greetings"

VOICES = [
    "alloy",
    "ash",
    "ballad",
    "beacon",
    "bossa",
    "cedar",
    "cinder",
    "coral",
    "delta",
    "echo",
    "gleam",
    "marin",
    "meridian",
    "quartz",
    "ripple",
    "sage",
    "shimmer",
    "stone",
    "tempo",
    "verse",
    "vesper",
    "willow",
]
# TALKING_SPEED = 1.0
MAX_RECORDINGS = 1

for voice in VOICES:
    for i in range(MAX_RECORDINGS):
        chat_client = OpenAILive(
            voice=voice,
            voice_prompt="You ONLY respond with concise greetings: hello, what's up, etc.",
            backend_prompt="You ONLY respond with concise greetings: hello, what's up, etc.",
        )
        chat_run_task = asyncio.create_task(chat_client.run())
        intro_audio = AudioChunk.empty(role="assistant", sample_rate=24000)

        while True:
            if not chat_client._session:
                print("NO SESSION")
                await asyncio.sleep(0.1)
                continue
            await chat_client.handle_user_audio(load_greeting(voice="alloy", idx=1))
            break

        while True:
            new_chunk = chat_client.pull_response_audio()
            print(new_chunk)
            if new_chunk and new_chunk.type == "audio_chunk":
                print("received audio chunk!")
                intro_audio += new_chunk
            elif intro_audio.duration() > 0.1:
                break  # model turn is over
            await asyncio.sleep(0.1)

        chat_run_task.cancel()

        out_path = f"{OUT_DIR}/{voice}_{i + 1}.npy"
        print(f"saving to {out_path}")
        np.save(out_path, intro_audio.samples)

NO SESSION
NO SESSION
NO SESSION
NO SESSION
NO SESSION
None
None
None
None
None
None
None
None
None
None
None
None
None
putting in queue: 0.1
type='audio_chunk' role='assistant' samples=array([  -4,   -4,   -4, ..., -186, -136, -169],
      shape=(2400,), dtype=int16) sample_rate=24000
received audio chunk!
putting in queue: 0.1
type='audio_chunk' role='assistant' samples=array([-225, -231, -177, ..., -154, -164, -143],
      shape=(2400,), dtype=int16) sample_rate=24000
received audio chunk!
putting in queue: 0.1
type='audio_chunk' role='assistant' samples=array([ -620, -1048, -1320, ...,  -340,  -356,  -338],
      shape=(2400,), dtype=int16) sample_rate=24000
received audio chunk!
putting in queue: 0.1
type='audio_chunk' role='assistant' samples=array([-251, -106,  -62, ...,   14,    3,   -4],
      shape=(2400,), dtype=int16) sample_rate=24000
received audio chunk!
putting in queue: 0.1
type='audio_chunk' role='assistant' samples=array([-3,  1,  6, ...,  0,  0,  0], shape=(2400,), 

In [5]:
from reachy_mini import ReachyMini
from assistant.audio_transports import LocalAudioTransport, ReachyAudioTransport

# reachy = ReachyMini()

for voice in VOICES:
    audio_transport = LocalAudioTransport(prebuffer_target_chunks=1)
    audio_transport.start()

    path = f"{OUT_DIR}/{voice}_1.npy"
    print(f"playing {path}...")
    loaded_audio = AudioChunk.from_file(path)

    audio_transport.push_to_speaker(loaded_audio)
    await asyncio.sleep(loaded_audio.duration())

    audio_transport.close()


playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/alloy_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/ash_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/ballad_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/beacon_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/bossa_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/cedar_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/cinder_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/coral_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/delta_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/echo_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greetings/gleam_1.npy...
playing /Users/jckpn-work/dev/assistant-redux/src/assistant/greet

||PaMacCore (AUHAL)|| Error on line 2523: err='-50', msg=Unknown Error
